In [9]:
import numpy as np
import requests

import requests
from datetime import datetime, timedelta
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
import csv
import time

#1. Getting prices
def prices(startd,endd,tic):
    info=[]
    key='&apikey=ZKMMTO1ATDBLXH2K'
    ticker='&symbol='+str(tic)
#    endpoint='function=TIME_SERIES_MONTHLY_ADJUSTED'
    endpoint='function=TIME_SERIES_DAILY_ADJUSTED'
    size='&outputsize=full'
    web='https://www.alphavantage.co/query?'
    url =web+endpoint+ticker+size+key

    r = requests.get(url)
    if r.status_code==200:
        print('connection successful')
        data = r.json() #need to convert to json to navigate
        r1=data.get('Time Series (Daily)', {})
        #r1=data.get('Monthly Adjusted Time Series', {})
        r2=data['Time Series (Daily)']
        #r2=data['Monthly Adjusted Time Series']
        
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info
    

    

def compute_log_returns(data, freq='daily'):
    """
    Computes log returns at specified frequency.
    
    Parameters:
    -----------
    data : list of lists
        Price data: [[ticker, date, adjusted_close], ...]
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    
    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return], ...]
    """
    sorted_data = sorted(data, key=lambda x: x[1])  # Sort by date (index 1)
    data_with_logret = []
    
    if freq == 'daily':
        for i in range(len(sorted_data)):
            if i == 0:
                data_with_logret.append(sorted_data[i] + [None])
            else:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[i-1][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
    
    elif freq == 'weekly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 7:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    elif freq == 'monthly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 30:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    elif freq == 'yearly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 360:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    return data_with_logret


#6. Download, compute returns, and export to CSV
def download_and_export(ticker, startd, endd, freq='daily', path=None):
    """
    Downloads price data, computes log and simple returns, and exports to CSV.

    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g. 'NVDA')
    startd : str
        Start date 'YYYY-MM-DD'
    endd : str
        End date 'YYYY-MM-DD'
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    path : str
        File path for CSV output. Defaults to '{ticker}_returns.csv'

    Returns:
    --------
    list of lists: [[ticker, date, price, log_return, simple_return], ...]
    First row is a header row.
    """
    if path is None:
        path = f"{ticker}_returns.csv"

    raw = prices(startd, endd, ticker)
    with_log = compute_log_returns(raw, freq=freq)
    with_both = compute_returns(with_log)

    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        for row in with_both:
            writer.writerow(row)

    print(f"Exported {len(with_both)-1} rows to {path}")
    return with_both


In [11]:
data_nvda = prices("2022-01-01", "2025-12-31", "NVDA")
time.sleep(5)
data_jpm = prices("2022-01-01", "2025-12-31", "JPM")
time.sleep(5)
data_aapl = prices("2022-01-01", "2025-12-31", "AAPL")
time.sleep(5)
data_spy = prices("2022-01-01", "2025-12-31", "SPY")

connection successful
connection successful
connection successful
connection successful


In [13]:
daily_nvda = compute_log_returns(data_nvda, freq='daily')
daily_jpm = compute_log_returns(data_jpm, freq='daily')
daily_aapl = compute_log_returns(data_aapl, freq='daily')
daily_spy = compute_log_returns(data_spy, freq='daily')

In [ ]:
def compute_returns(data):
    """
    Computes simple and log returns from compute_log_returns output.
    
    Parameters:
    -----------
    data : list of lists
        [[ticker, date, adjusted_close, log_return], ...]
    
    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return, simple_return], ...]
    First row is a header row.
    """
    header = ["ticker", "date", "price", "log_return", "simple_return"]
    result = [header]
    for i in range(len(data)):
        if i == 0 or data[i-1][2] is None:
            result.append(data[i][:4] + [None])
        else:
            price_current = float(data[i][2])
            price_previous = float(data[i-1][2])
            simple_ret = (price_current - price_previous) / price_previous
            result.append(data[i][:4] + [simple_ret])
    return result


In [ ]:
nvda = compute_returns(daily_nvda)
jpm = compute_returns(daily_jpm)
aapl = compute_returns(daily_aapl)
spy = compute_returns(daily_spy)

nvda

In [ ]:
nvda = download_and_export("NVDA", "2022-01-01", "2025-12-31", freq="daily", path="nvda_returns.csv")
time.sleep(5)
jpm  = download_and_export("JPM",  "2022-01-01", "2025-12-31", freq="daily", path="jpm_returns.csv")
time.sleep(5)
aapl = download_and_export("AAPL", "2022-01-01", "2025-12-31", freq="daily", path="aapl_returns.csv")
time.sleep(5)
spy  = download_and_export("SPY",  "2022-01-01", "2025-12-31", freq="daily", path="spy_returns.csv")


In [ ]:



daily_nvda

connection successful


[['NVDA', '2022-01-03', '30.0620151718419', None],
 ['NVDA', '2022-01-04', '29.232642488073', np.float64(-0.027976442072136454)],
 ['NVDA', '2022-01-05', '27.5499441188381', np.float64(-0.05928547108128601)],
 ['NVDA', '2022-01-06', '28.1228200760984', np.float64(0.020580841872730016)],
 ['NVDA', '2022-01-07', '27.1936432185909', np.float64(-0.03359810832549037)],
 ['NVDA', '2022-01-10', '27.3463436044111', np.float64(0.005599590088429807)],
 ['NVDA', '2022-01-11', '27.7625270089016', np.float64(0.015104331197398846)],
 ['NVDA', '2022-01-12', '27.9441706051061', np.float64(0.006521450660296857)],
 ['NVDA', '2022-01-13', '26.5229591710666', np.float64(-0.052197871023708764)],
 ['NVDA', '2022-01-14', '26.8892404886877', np.float64(0.013715483048363453)],
 ['NVDA', '2022-01-18', '25.8522751235424', np.float64(-0.03932761516284122)],
 ['NVDA', '2022-01-19', '25.0179122310866', np.float64(-0.03280655204192495)],
 ['NVDA', '2022-01-20', '24.1027079579025', np.float64(-0.03726785997302651)],
